In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import  Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\PC\AppData\Local\Temp\ipykernel_11080\2954398867.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
docs=PyPDFLoader(r"C:\Users\PC\Downloads\Microsoft 2025 Annual Report.pdf").load()
chunks=RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=20).split_documents(docs)
print(len(chunks))

712


In [3]:
for i, doc in enumerate(chunks):
    if not isinstance(doc.page_content, str):
        print(f"Index {i} par content string nahi hai: {type(doc.page_content)}")

In [4]:
embeding=HuggingFaceEmbeddings(model_name='all-miniLm-L6-v2')


C:\Users\PC\AppData\Local\Temp\ipykernel_11080\3663232769.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeding=HuggingFaceEmbeddings(model_name='all-miniLm-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
vectorestore=Chroma.from_documents(documents=chunks,embedding=embeding)

In [6]:
retriver=vectorestore.as_retriever()

In [50]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model='smollm:135m')



In [34]:
#print(docs[1].page_content)

In [37]:
from collections import defaultdict

# -----------------------------
# 1. Generate Query Variants
# -----------------------------
def generate_query_variants(query, llm, n=4):

    prompt = f"""
You are a helpful AI assistant.

Generate {n} different search queries for the following question.

Question:
{query}

Return only one query per line.
"""

    response = llm.invoke(prompt)

    variants = [
        line.strip()
        for line in response.content.split("\n")
        if line.strip()
    ]

    return variants[:n]


# -----------------------------
# 2. Reciprocal Rank Fusion
# -----------------------------
def reciprocal_rank_fusion(results, k=60):

    scores = defaultdict(float)
    documents = {}

    for docs in results:
        for rank, doc in enumerate(docs):

            # unique key
            key = (
                doc.page_content,
                tuple(sorted(doc.metadata.items()))
            )

            scores[key] += 1 / (rank + k + 1)

            documents[key] = doc

    reranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    fused_docs = [
        documents[key]
        for key, _ in reranked
    ]

    return fused_docs


# -----------------------------
# 3. Generate Final Answer
# -----------------------------
def generate_answer(query, docs, llm):

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    prompt = f"""
Answer the question only from the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content


# -----------------------------
# 4. Complete RAG Fusion
# -----------------------------
def rag_fusion(query, retriever, llm, n_queries=4, top_k=5):

    # Original + Variants
    variants = generate_query_variants(
        query,
        llm,
        n=n_queries
    )

    all_queries = [query] + variants

    print("Generated Queries:")
    for q in all_queries:
        print("-", q)

    # Retrieve
    retrieved_results = []

    for q in all_queries:
        docs = retriever.invoke(q)
        retrieved_results.append(docs)

    # Fuse
    fused_docs = reciprocal_rank_fusion(retrieved_results)

    # Final Answer
    answer = generate_answer(
        query,
        fused_docs[:top_k],
        llm
    )

    return answer, fused_docs[:top_k]

In [56]:
query = "what is Annual report 2025 ?"

answer, docs = rag_fusion(
    query=query,
    retriever=retriver,
    llm=llm,
    n_queries=4,
    top_k=5
)

print("Answer:")
print(answer)
print('*'*50)

Generated Queries:
- what is Annual report 2025 ?
- Here's an example of what the query might look like in a single line, with the correct answer:
- ```python
- query = "What is Annual Report 2025 ?"
- ```
Answer:
Annual Report 2025 is an annual report that summarizes the financial performance of Microsoft 365 customers across the United States and Canada over a year. It provides a snapshot of the company's financial health, highlighting areas for improvement and opportunities to increase profitability.

The Annual Report 2025 is released quarterly by Microsoft on a regular basis (usually every two months) and includes detailed information about the company's financials, including:

1. **Financial performance**: The report provides an overview of the company's financial health, including revenue growth, expenses, cash flow, and profitability metrics such as gross margin, operating margin, and return on equity (ROE).
2. **Growth rate**: The report highlights the company's rapid growth i

In [44]:
Documents
    ↓
Embedding
    ↓
Vector Store (FAISS/Chroma)
    ↓
Retriever
    ↓
RAG Fusion
    ↓
ChatOllama
    ↓
Answer

IndentationError: unexpected indent (266813609.py, line 2)